# EDA Exploratorio – Similitud Coseno entre Etiquetas (0 vs 4)

Este notebook realiza un análisis **exploratorio** para evaluar la **consistencia semántica de las etiquetas 0 (negativo) y 4 (positivo)** utilizando similitud coseno entre centroides TF-IDF.

Este análisis **no está asociado a ningún modelo predictivo**, y se utiliza únicamente con fines descriptivos dentro del EDA.


## 1. Carga de datos

In [ ]:

import pandas as pd

df = pd.read_csv('/mnt/data/training.1600000.processed.noemoticon.csv', 
                 encoding='latin-1', header=None)
df.columns = ['polarity','id','date','query','user','text']

df = df[df['polarity'].isin([0,4])]
df.head()


## 2. Limpieza básica y lematización

In [ ]:

import re
import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner","parser"])

def clean_tweet(t):
    t = re.sub(r"http\S+","", t)
    t = re.sub(r"@[A-Za-z0-9_]+","", t)
    t = re.sub(r"#","", t)
    t = re.sub(r"[^a-zA-Z ]","", t)
    return t.lower()

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc if not token.is_stop and token.lemma_ != "-PRON-"])

df["clean"] = df["text"].astype(str).apply(clean_tweet)
df["lemma"] = df["clean"].apply(lemmatize_text)
df.head()


## 3. Vectorización TF-IDF Exploratoria

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_eda = TfidfVectorizer(max_features=10000, stop_words='english')
X_eda_tfidf = tfidf_eda.fit_transform(df["lemma"])


## 4. Cálculo de centroides por clase

In [ ]:

import numpy as np

mask_0 = df["polarity"] == 0
mask_4 = df["polarity"] == 4

X_0 = X_eda_tfidf[mask_0]
X_4 = X_eda_tfidf[mask_4]

centroid_0 = X_0.mean(axis=0)
centroid_4 = X_4.mean(axis=0)

centroid_0.shape, centroid_4.shape


## 5. Similitud coseno entre etiquetas

In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

sim_eda = cosine_similarity(centroid_0, centroid_4)[0][0]
print(f"Similitud coseno exploratoria entre etiquetas 0 y 4: {sim_eda:.4f}")


## 6. Visualización

In [ ]:

import matplotlib.pyplot as plt

mat = [[1, sim_eda],[sim_eda, 1]]

plt.figure(figsize=(4,3))
plt.imshow(mat, cmap="viridis")
plt.xticks([0,1], ["Negativo (0)", "Positivo (4)"])
plt.yticks([0,1], ["Negativo (0)", "Positivo (4)"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{mat[i][j]:.2f}", ha="center", va="center", color="white")
plt.title("Similitud coseno entre centroides (EDA)")
plt.colorbar()
plt.tight_layout()
plt.show()



## 7. Conclusión (EDA)

El valor de similitud coseno obtenido entre los centroides de las clases negativa (0) y positiva (4) indica el grado de proximidad semántica promedio entre ambos grupos en el espacio TF-IDF exploratorio.

Dado que este análisis se realiza **antes del entrenamiento de cualquier modelo**, su objetivo es exclusivamente evaluar la **coherencia global de las etiquetas del dataset** y no anticipar rendimiento predictivo.
